
# 🎨 ClawSouls — Avatar API Server (FastAPI + Cloudflared)

Sobe um servidor **FastAPI** no Colab que gera avatares por requisição HTTP.

**Fluxo:**
1. Rode este notebook (com GPU)
2. Copie a URL pública do Cloudflared
3. Envie POST `/generate` com os atributos da soul
4. Receba a imagem gerada em base64

---



## Pré-requisitos

- `Runtime > Change runtime type > T4 GPU`
- Não precisa do repositório clonado (tudo é self-contained)


In [ ]:

# ═══════════════════════════════════════════════════════════
# TOGGLE DE MODELO
# ═══════════════════════════════════════════════════════════

MODELO = "turbo"          # ← "sdxl" ou "turbo"
SECRET_TOKEN = "cs-secret-2026"  # ← Troque para algo seguro!
TOTAL_STEPS = 8             # Override global de steps (turbo=8, sdxl=25)
TOTAL_GUIDANCE = 0.0        # Override global de guidance (turbo=0.0, sdxl=7.5)

# HF Token: coloque aqui ou passe como env var HF_TOKEN=...
HF_TOKEN = ""
if HF_TOKEN:
    import os as _os
    _os.environ["HF_TOKEN"] = HF_TOKEN

CATALOGO = {
    "turbo": {
        "model_id": "T5B/Z-Image-Turbo-FP8",
        "width": 1024, "height": 1024, "variant": None,
        "crop_width": 512, "crop_height": 768,
        "default_steps": 8, "default_guidance": 0.0,
    },
    "sdxl": {
        "model_id": "stabilityai/stable-diffusion-xl-base-1.0",
        "width": 512, "height": 768, "variant": "fp16",
        "crop_width": 0, "crop_height": 0,
        "default_steps": 25, "default_guidance": 7.5,
    },
}

assert MODELO in CATALOGO, f"Use: {list(CATALOGO.keys())}"
cfg = CATALOGO[MODELO]
MODEL_ID = cfg["model_id"]

print(f"🔧 Modelo: {MODELO} ({MODEL_ID})")
print(f"   Steps: {TOTAL_STEPS}, Guidance: {TOTAL_GUIDANCE}")
if HF_TOKEN:
    print(f"   HF_TOKEN: {HF_TOKEN[:8]}...")


In [ ]:

# Escreve o servidor num arquivo .py (servidor thin — só recebe prompt e gera)
# O prompt é montado do lado do agente; o servidor é burro e direto.
# O conteúdo é copiado verbatim para /content/server.py na célula seguinte.

server_code = '''
from urllib.parse import parse_qs
import io, base64, time, os, re
from datetime import datetime
from typing import Optional
from PIL import Image

from pydantic import BaseModel
from fastapi import FastAPI, HTTPException, Request
import torch
from diffusers import AutoPipelineForText2Image

# ── Config (injetadas pelo notebook via env vars) ──────────
MODELO = os.environ.get('CLAWSOULS_MODELO', 'turbo')
SECRET_TOKEN = os.environ.get('CLAWSOULS_SECRET', 'cs-secret-2026')
TOTAL_STEPS = int(os.environ.get('CLAWSOULS_STEPS', '8'))
TOTAL_GUIDANCE = float(os.environ.get('CLAWSOULS_GUIDANCE', '0.0'))
MODEL_ID = os.environ.get('CLAWSOULS_MODEL_ID', 'T5B/Z-Image-Turbo-FP8')
MODEL_WIDTH = int(os.environ.get('CLAWSOULS_WIDTH', '1024'))
MODEL_HEIGHT = int(os.environ.get('CLAWSOULS_HEIGHT', '1024'))
CROP_WIDTH = int(os.environ.get('CLAWSOULS_CROP_WIDTH', '512'))
CROP_HEIGHT = int(os.environ.get('CLAWSOULS_CROP_HEIGHT', '768'))

DEFAULT_NEGATIVE = (
    'blurry, low quality, deformed, ugly, duplicate, disfigured, '
    'bad anatomy, bad proportions, extra limbs, mutated hands, '
    'text, watermark, signature, logo, '
    'photorealistic, 3d render, '
    'nude, NSFW, gore'
)

# ── FastAPI App ────────────────────────────────────────
class GenerateRequest(BaseModel):
    name: Optional[str] = 'unnamed'
    custom_prompt: str
    steps: Optional[int] = None
    guidance: Optional[float] = None
    custom_negative_prompt: Optional[str] = None

app = FastAPI(title='ClawSouls Avatar API', version='3.0-turbo')

def _check_token(request: Request) -> bool:
    auth = request.headers.get('Authorization', '')
    if auth == 'Bearer ' + SECRET_TOKEN:
        return True
    qs = parse_qs(request.url.query)
    if qs.get('token', [''])[0] == SECRET_TOKEN:
        return True
    if request.headers.get('X-Token') == SECRET_TOKEN:
        return True
    return False

def center_crop(image, target_w, target_h):
    """Center crop a PIL image to target dimensions."""
    w, h = image.size
    if w < target_w or h < target_h:
        scale = max(target_w / w, target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)
        image = image.resize((new_w, new_h), resample=Image.LANCZOS)
        w, h = new_w, new_h
    left = (w - target_w) // 2
    top = (h - target_h) // 2
    return image.crop((left, top, left + target_w, top + target_h))

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODELO, 'model_id': MODEL_ID}

@app.get('/models')
def list_models():
    return {'available': {}, 'current': MODELO}

@app.post('/generate')
async def generate(request: Request, req: GenerateRequest):
    if not _check_token(request):
        raise HTTPException(status_code=401, detail='Unauthorized')
    prompt = req.custom_prompt
    negative = req.custom_negative_prompt or DEFAULT_NEGATIVE
    steps = req.steps if req.steps else TOTAL_STEPS
    guidance = req.guidance if req.guidance else TOTAL_GUIDANCE

    if MODELO == 'turbo':
        guidance = 0.0

    print(f"[{datetime.now().strftime('%H:%M:%S')}] Generating: {req.name} ({steps} steps, guidance={guidance})")
    start = time.time()
    generator = torch.Generator(device='cuda').manual_seed(int(time.time() * 1000) % (2**32))

    if MODELO == 'turbo':
        image = pipe(
            prompt=prompt,
            num_inference_steps=steps,
            generator=generator,
        ).images[0]
    else:
        image = pipe(
            prompt=prompt, negative_prompt=negative,
            num_inference_steps=steps, guidance_scale=guidance,
            width=MODEL_WIDTH, height=MODEL_HEIGHT, generator=generator,
        ).images[0]

    if CROP_WIDTH and CROP_HEIGHT:
        image = center_crop(image, CROP_WIDTH, CROP_HEIGHT)

    elapsed = time.time() - start
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode('utf-8')
    safe_name = ''.join(c if c.isalnum() or c in '._-' else '_' for c in req.name.lower().strip())
    print(f"   ✅ {req.name} → {elapsed:.1f}s")
    return {
        'name': req.name, 'slug': safe_name,
        'prompt': prompt, 'negative_prompt': negative,
        'seed': int(generator.initial_seed()),
        'steps': steps, 'guidance': guidance, 'model': MODELO,
        'elapsed_s': round(elapsed, 1), 'image_base64': img_b64,
    }

# ── Lifespan (model loading) ─────────────────────────────
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app):
    global pipe

    hf_token = os.environ.get('HF_TOKEN', '')
    if not hf_token:
        token_path = os.path.expanduser('~/.cache/huggingface/token')
        if os.path.exists(token_path):
            with open(token_path) as f:
                hf_token = f.read().strip()

    print(f"Loading model: {MODEL_ID}")
    print(f"  model={MODELO}, token={'set' if hf_token else 'NONE'}")

    if MODELO == 'turbo':
        try:
            from diffusers import ZImagePipeline
            print(f"  Loading ZImagePipeline from {MODEL_ID}...")
            pipe = ZImagePipeline.from_pretrained(
                MODEL_ID,
                torch_dtype=torch.bfloat16,
                use_safetensors=True,
                token=hf_token if hf_token else None,
            )
            pipe.to('cuda')
            print('  ✅ Modelo Z-Image-Turbo carregado!')
        except Exception as e:
            print(f'  ❌ ZImagePipeline falhou: {e}')
            raise
    else:
        attempt_variants = ['fp16', None]
        pipe = None
        for attempt_variant in attempt_variants:
            try:
                print(f"  Trying variant={attempt_variant}...")
                pipe = AutoPipelineForText2Image.from_pretrained(
                    MODEL_ID,
                    torch_dtype=torch.float16,
                    variant=attempt_variant,
                    use_safetensors=True,
                    token=hf_token if hf_token else None,
                )
                pipe.enable_attention_slicing()
                if hasattr(pipe, 'enable_vae_tiling'):
                    pipe.enable_vae_tiling()
                pipe.to('cuda')
                if hasattr(pipe, 'vae'):
                    pipe.vae.to('cuda', torch.float16)
                print(f'  ✅ Modelo SDXL carregado (variant={attempt_variant})')
                break
            except Exception as e:
                print(f'  ⚠️  Falhou com variant={attempt_variant}: {e}')
                if attempt_variant is None:
                    raise

    if pipe is None:
        raise RuntimeError('Não foi possível carregar o modelo!')

    yield
    del pipe
    torch.cuda.empty_cache()
    print('🛑 Servidor encerrado')

app.router.lifespan_context = lifespan
'''


In [ ]:

# Silenciar warnings
import warnings, os
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

!pip install -q fastapi uvicorn[standard] pydantic pillow
!pip install -q git+https://github.com/huggingface/diffusers transformers accelerate torch torchvision safetensors huggingface_hub

# cloudflared não está no PyPI — baixa o binário direto do GitHub
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Dependências instaladas!")


In [ ]:

# ── Monta server.py final com as configurações do notebook ──

# Gera server.py final no /content/ substituindo placeholders
_final_server = server_code
_final_server = _final_server.replace(
    "MODELO = os.environ.get('CLAWSOULS_MODELO', 'turbo')",
    f"MODELO = os.environ.get('CLAWSOULS_MODELO', '{MODELO}')"
)
_final_server = _final_server.replace(
    "SECRET_TOKEN = os.environ.get('CLAWSOULS_SECRET', 'cs-secret-2026')",
    f"SECRET_TOKEN = os.environ.get('CLAWSOULS_SECRET', '{SECRET_TOKEN}')"
)
_final_server = _final_server.replace(
    "TOTAL_STEPS = int(os.environ.get('CLAWSOULS_STEPS', '8'))",
    f"TOTAL_STEPS = int(os.environ.get('CLAWSOULS_STEPS', '{TOTAL_STEPS}'))"
)
_final_server = _final_server.replace(
    "TOTAL_GUIDANCE = float(os.environ.get('CLAWSOULS_GUIDANCE', '0.0'))",
    f"TOTAL_GUIDANCE = float(os.environ.get('CLAWSOULS_GUIDANCE', '{TOTAL_GUIDANCE}'))"
)
_final_server = _final_server.replace(
    "MODEL_ID = os.environ.get('CLAWSOULS_MODEL_ID', 'T5B/Z-Image-Turbo-FP8')",
    f"MODEL_ID = os.environ.get('CLAWSOULS_MODEL_ID', '{MODEL_ID}')"
)
_final_server = _final_server.replace(
    "MODEL_WIDTH = int(os.environ.get('CLAWSOULS_WIDTH', '1024'))",
    f"MODEL_WIDTH = int(os.environ.get('CLAWSOULS_WIDTH', '{cfg['width']}'))"
)
_final_server = _final_server.replace(
    "MODEL_HEIGHT = int(os.environ.get('CLAWSOULS_HEIGHT', '1024'))",
    f"MODEL_HEIGHT = int(os.environ.get('CLAWSOULS_HEIGHT', '{cfg['height']}'))"
)
_final_server = _final_server.replace(
    "CROP_WIDTH = int(os.environ.get('CLAWSOULS_CROP_WIDTH', '512'))",
    f"CROP_WIDTH = int(os.environ.get('CLAWSOULS_CROP_WIDTH', '{cfg.get('crop_width', 512)}'))"
)
_final_server = _final_server.replace(
    "CROP_HEIGHT = int(os.environ.get('CLAWSOULS_CROP_HEIGHT', '768'))",
    f"CROP_HEIGHT = int(os.environ.get('CLAWSOULS_CROP_HEIGHT', '{cfg.get('crop_height', 768)}'))"
)

# Escreve server.py no /content/
with open('/content/server.py', 'w') as f:
    f.write(_final_server)

print('✅ server.py escrito em /content/')


In [ ]:

# Inicia o servidor FastAPI
import subprocess
import time

env = {
    'CLAWSOULS_MODELO': MODELO,
    'CLAWSOULS_SECRET': SECRET_TOKEN,
    'CLAWSOULS_STEPS': str(TOTAL_STEPS),
    'CLAWSOULS_GUIDANCE': str(TOTAL_GUIDANCE),
    'CLAWSOULS_MODEL_ID': MODEL_ID,
    'CLAWSOULS_WIDTH': str(cfg['width']),
    'CLAWSOULS_HEIGHT': str(cfg['height']),
    'CLAWSOULS_CROP_WIDTH': str(cfg.get('crop_width', 512)),
    'CLAWSOULS_CROP_HEIGHT': str(cfg.get('crop_height', 768)),
}

proc = subprocess.Popen(
    ['python', '/content/server.py'],
    env={**dict(__import__('os').environ), **env},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print('🚀 Servidor iniciado...')
time.sleep(3)

# Testa healthcheck
import urllib.request
import json as _json
try:
    with urllib.request.urlopen('http://localhost:8000/health') as resp:
        health = _json.loads(resp.read().decode())
        print(f"✅ Health: {health}")
except Exception as e:
    print(f'⚠️  Healthcheck falhou: {e}')


---

## Túnel Cloudflared


In [ ]:

import subprocess
import re

# Usa o token da conta Cloudflare fornecido
CLOUDFLARE_TOKEN = "eyJhIjoiNmIxNmYwMzUwNjM5NWFhNjBjZjk2NzY0MDA2Y2I0MGUiLCJ0IjoiYzA1MzQ2NGEtNzBhYy00MmUwLWFkMjQtZDdiMDFkYzBmOGZhIiwicyI6Ik1qbGtOekEyWVRjdFkyWXdOQzAwTXpGa0xXSTNZV1V0WkdJek5HVTJNRGhrT0RVeCJ9"

# Inicia cloudflared tunnel usando o token da conta
cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', 'run', '--token', CLOUDFLARE_TOKEN],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Espera a URL do tunnel aparecer nos logs
tunnel_url = None
print('⏳ Esperando URL do tunnel...')
for i in range(30):
    line = cloudflared_proc.stdout.readline().decode('utf-8', errors='replace')
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print()
    print('=' * 60)
    print('🌐 TÚNEL ATIVO!')
    print(f'📎 URL pública: {tunnel_url}')
    print()
    print('Endpoints:')
    print(f'  GET  {tunnel_url}/health')
    print(f'  GET  {tunnel_url}/models')
    print(f'  POST {tunnel_url}/generate')
    print('=' * 60)
    print()
    print('⚠️  TOKEN: ' + SECRET_TOKEN)
    print()
    print('📋 Copie esta URL e cole aqui no chat para eu usar!')
else:
    print('⚠️  Não foi possível obter a URL do tunnel.')
    print('   Verifique: cloudflared_proc.wait()')


---

## Como usar

Cole a URL do tunnel aqui no chat. Eu monto o prompt e faço a requisição!

**Exemplo mínimo:**

```
POST /generate?token=cs-secret-2026
{"custom_prompt": "a cyberpunk mage with bionic arms, neon glow, detailed face"}
```

**Exemplo completo:**

```
POST /generate?token=cs-secret-2026
{"name": "Mage Cyberpunk", "custom_prompt": "a grizzled 60-year-old mage..."}
```

**Notas:**
- Turbo (default): 1024×1024 gerado, cropado para 512×768 bust portrait
- ~2-4 segundos por imagem
- Sem guidance (CFG=0) — prompt deve ser bem descritivo
